# 01 · Data team: reproduce the baseline

Goal: run Meta's pretrained model on the **test users**, get **CLER** per user, and
make a chart for the demo. Your mean CLER should match the baseline in the README.
If it does, the pipeline works.

**CLER** (classification error rate) is Meta's metric for this task. Every gesture
the model fires is lined up in time with the true gestures. CLER is the share that
got the wrong name, averaged over the 9 gesture types. **Lower is better.**
*Gesture accuracy = 1 − CLER.*

*Model and data: Meta, generic-neuromotor-interface (CC BY-NC 4.0).*

In [ ]:
# --- Setup (run once) -------------------------------------------------------
# Local: you've already run `pip install -r requirements.txt` in this repo, so skip ahead.
# Colab: this cell clones the repo, installs it, and points at the shared data.
import os, sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/sandeepvijayarao09/wearable-ai-build-day.git"  # organizers fill in
    if not os.path.exists("/content/repo"):
        !git clone -q {REPO_URL} /content/repo
    %cd /content/repo
    !pip install -q -r requirements.txt
    # Shared data: in Drive, right-click the shared "emg_data" folder ->
    # Organize -> Add shortcut -> My Drive. Then:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["EMG_DATA_DIR"] = "/content/drive/MyDrive/emg_data"
    os.environ["EMG_MODELS_DIR"] = "/content/drive/MyDrive/emg_data/emg_models"
else:
    # Run from the repo root so `emg_kit` imports.
    if not os.path.exists("emg_kit") and os.path.exists("../emg_kit"):
        os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from emg_kit.core import (all_test_users, detect_events, evaluate, load_model,
                          load_recording, predict, test_partitions)

model = load_model()   # Meta's checkpoint from ~/emg_models (or $EMG_MODELS_DIR)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")

have, want = list(test_partitions()), all_test_users()
print(f"{len(have)} of {len(want)} test users on this machine: {have}")
if len(have) < len(want):
    print("Point EMG_DATA_DIR at the full shared dataset for all 10.")

## 1. One user, step by step

Before running everyone, look at what the model actually outputs for one recording.

In [ ]:
user = have[0]
rec = load_recording(*test_partitions()[user], user=user)
probs, prob_times = predict(model, rec.emg, rec.times)
print("EMG in:   ", rec.emg.shape, "(channels, samples at 2 kHz)")
print("probs out:", probs.shape, "(gestures, steps at 200 Hz)")

events = detect_events(probs, prob_times)   # threshold 0.35, 50 ms debounce
print(f"model fired {len(events)} gestures; there were {len(rec.prompts)} real ones")
events.head()

In [ ]:
from emg_kit.core import GESTURES
T0, SPAN = 60, 8
t = prob_times - rec.times[0]
w = (t >= T0) & (t < T0 + SPAN)
fig, ax = plt.subplots(figsize=(13, 4.5))
for i, g in enumerate(GESTURES):
    ax.plot(t[w], probs[i, w] + i, lw=1, label=g)
for _, r in rec.prompts.iterrows():
    x = r.time - rec.times[0]
    if T0 <= x < T0 + SPAN:
        i = GESTURES.index(r["name"])
        ax.plot([x, x], [i, i + 1], color="k", lw=2)
ax.set_yticks([i + 0.5 for i in range(9)], GESTURES)
ax.set_xlabel("time (s)")
ax.set_title("Predicted probability per gesture (colour) vs. true gesture (black tick)")
plt.tight_layout()

## 2. All test users -> the baseline number

Each test user is 25-42 minutes of EMG. Expect ~30 s per user on a fast laptop and
~5 minutes per user on Colab's CPU. On Colab, run the 3 users in the small subset
(target 0.1197) and leave all 10 for a laptop.

In [ ]:
results = evaluate(model)
results.to_csv("results_baseline.csv", index=False)
print(f"\nMEAN CLER over {len(results)} users: {results.cler.mean():.4f}"
      f"   (gesture accuracy {1 - results.cler.mean():.1%})")
results

## 3. Per-user chart (goes into the demo at 1:15)

In [ ]:
r = results.sort_values("accuracy")
fig, ax = plt.subplots(figsize=(8, 0.45 * len(r) + 1.2))
ax.barh(r.user, r.accuracy * 100, color="#4C6EF5")
ax.axvline(results.accuracy.mean() * 100, color="k", ls="--", lw=1,
           label=f"mean {results.accuracy.mean():.1%}")
for y, v in zip(r.user, r.accuracy):
    ax.text(v * 100 + 0.3, y, f"{v:.1%}", va="center", fontsize=9)
ax.set_xlabel("gesture accuracy (1 − CLER), %")
ax.set_xlim(max(0, r.accuracy.min() * 100 - 10), 100)
ax.set_title("Meta's pretrained model, held-out test users")
ax.legend(loc="lower right", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("per_user_accuracy.png", dpi=160)
print("saved per_user_accuracy.png -> hand this to the Demo team")

## Stretch goals

- **Which gestures are hardest?** Use `generic_neuromotor_interface.cler.compute_confusion_matrix`
  on one user and plot it as a heatmap.
- **Pick demo clips.** Find 5–10 second windows where the model gets every gesture right
  (and one where it doesn't, which is honest and more interesting). Send `(user, start second)`
  to the Demo team. `make_clips.py` shows how clips are cut.
- **Threshold sweep.** CLER uses a 0.35 threshold. What happens at 0.25 or 0.5?